# Network Motif Finding with GraphFrames

This notebook covers GraphFrames' motif finding feature using **Apache Spark 4.0**. We perform pattern matching on a property graph representing a Stack Exchange site, using PySpark to build a property graph and then mine it for network motifs by combining both graph and relational queries.

**Prerequisites**: Complete the [Data Setup Tutorial](https://graphframes.io/03-tutorials/03-data-setup.html) to download and prepare the Stack Exchange dataset before running this notebook.

## What are Graphlets and Network Motifs?

Graphlets are small, connected subgraphs of a larger graph. Network motifs are recurring patterns in complex networks that are significantly more frequent than in random networks. They are the building blocks of complex networks and can be used to understand the structure and function of networks.

We are going to mine motifs using Stack Exchange data. The Stack Exchange network is a complex network of users, posts, votes, badges, and tags. We will use GraphFrames to build a property graph from the Stack Exchange data dump and then use GraphFrames' motif finding feature to find network motifs in the graph.

## Setup and Data Loading

In [ ]:
import pyspark.sql.functions as F
from graphframes import GraphFrame
from pyspark.sql import DataFrame, SparkSession

# Initialize a SparkSession
spark: SparkSession = (
    SparkSession.builder.appName("Stack Overflow Motif Analysis")
    .config("spark.sql.caseSensitive", True)
    .getOrCreate()
)
spark.sparkContext.setCheckpointDir("/tmp/graphframes-checkpoints/motif")

# Change me if you download a different stackexchange site
STACKEXCHANGE_SITE = "stats.meta.stackexchange.com"
BASE_PATH = f"python/graphframes/tutorials/data/{STACKEXCHANGE_SITE}"

Load the nodes and edges. We repartition for parallelism and cache for performance.

In [ ]:
NODES_PATH: str = f"{BASE_PATH}/Nodes.parquet"
nodes_df: DataFrame = spark.read.parquet(NODES_PATH)
nodes_df = nodes_df.repartition(50).checkpoint().cache()

EDGES_PATH: str = f"{BASE_PATH}/Edges.parquet"
edges_df: DataFrame = spark.read.parquet(EDGES_PATH)
edges_df = edges_df.repartition(50).checkpoint().cache()

## Explore the Graph

In [ ]:
# What kind of nodes do we have?
(
    nodes_df.select("id", F.col("Type").alias("Node Type"))
    .groupBy("Node Type")
    .count()
    .orderBy(F.col("count").desc())
    .withColumn("count", F.format_number(F.col("count"), 0))
    .show()
)

In [ ]:
# What kind of edges do we have?
(
    edges_df.select("src", "dst", F.col("relationship").alias("Edge Type"))
    .groupBy("Edge Type")
    .count()
    .orderBy(F.col("count").desc())
    .withColumn("count", F.format_number(F.col("count"), 0))
    .show()
)

## Create the GraphFrame

In [ ]:
g = GraphFrame(nodes_df, edges_df)

print(f"Vertices: {g.vertices.count():,}")
print(f"Edges: {g.edges.count():,}")
print(f"Node columns: {g.vertices.columns}")

## Validate the Graph

In [ ]:
# Sanity test that all edges have valid ids
edge_count = g.edges.count()
valid_edge_count = (
    g.edges.join(g.vertices, on=g.edges.src == g.vertices.id)
    .select("src", "dst", "relationship")
    .join(g.vertices, on=g.edges.dst == g.vertices.id)
    .count()
)
assert edge_count == valid_edge_count, f"Edge count {edge_count} != valid edge count {valid_edge_count}"
print(f"Edge count: {edge_count:,} == Valid edge count: {valid_edge_count:,}")

## Structural Motifs

Let's look for simple motifs. The `g.find()` method takes a Cypher-like pattern and returns a DataFrame with fields for each node and edge label in the pattern.

### G4: Continuous Triangles

In [ ]:
# G4: Continuous Triangles
paths = g.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)")

graphlet_type_df = paths.select(
    F.col("a.Type").alias("A_Type"),
    F.col("e1.relationship").alias("(a)-[e1]->(b)"),
    F.col("b.Type").alias("B_Type"),
    F.col("e2.relationship").alias("(b)-[e2]->(c)"),
    F.col("c.Type").alias("C_Type"),
    F.col("e3.relationship").alias("(c)-[e3]->(a)"),
)

graphlet_count_df = (
    graphlet_type_df.groupby(
        "A_Type", "(a)-[e1]->(b)", "B_Type", "(b)-[e2]->(c)", "C_Type", "(c)-[e3]->(a)"
    )
    .count()
    .orderBy(F.col("count").desc())
    .withColumn("count", F.format_number(F.col("count"), 0))
)
graphlet_count_df.show()

### G5: Divergent Triangles

In [ ]:
# G5: Divergent Triangles
paths = g.find("(a)-[e1]->(b); (a)-[e2]->(c); (c)-[e3]->(b)")

graphlet_type_df = paths.select(
    F.col("a.Type").alias("A_Type"),
    F.col("e1.relationship").alias("(a)-[e1]->(b)"),
    F.col("b.Type").alias("B_Type"),
    F.col("e2.relationship").alias("(a)-[e2]->(c)"),
    F.col("c.Type").alias("C_Type"),
    F.col("e3.relationship").alias("(c)-[e3]->(b)"),
)

graphlet_count_df = (
    graphlet_type_df.groupby(
        "A_Type", "(a)-[e1]->(b)", "B_Type", "(a)-[e2]->(c)", "C_Type", "(c)-[e3]->(b)"
    )
    .count()
    .orderBy(F.col("count").desc())
    .withColumn("count", F.format_number(F.col("count"), 0))
)
graphlet_count_df.show()

## Property Graph Motifs

We can do more than count motifs by type. Let's explore 4-node directed 3-paths and use property-level aggregation to find interesting patterns.

In [ ]:
# G17/G30: A directed 3-path
paths = g.find("(a)-[e1]->(b); (b)-[e2]->(c); (d)-[e3]->(c)")

graphlet_type_df = paths.select(
    F.col("a.Type").alias("A_Type"),
    F.col("e1.relationship").alias("(a)-[e1]->(b)"),
    F.col("b.Type").alias("B_Type"),
    F.col("e2.relationship").alias("(b)-[e2]->(c)"),
    F.col("c.Type").alias("C_Type"),
    F.col("e3.relationship").alias("(d)-[e3]->(c)"),
    F.col("d.Type").alias("D_Type"),
)
graphlet_count_df = (
    graphlet_type_df.groupby(
        "A_Type", "(a)-[e1]->(b)", "B_Type", "(b)-[e2]->(c)",
        "C_Type", "(d)-[e3]->(c)", "D_Type",
    )
    .count()
    .orderBy(F.col("count").desc())
    .withColumn("count", F.format_number(F.col("count"), 0))
)
graphlet_count_df.show(20)
print(f"Total distinct motif types: {graphlet_count_df.count()}")

### Correlation Analysis: Votes on Linked Questions

In [ ]:
# Filter to votes cast for linked questions
linked_vote_paths = paths.filter(
    (F.col("a.Type") == "Vote") &
    (F.col("e1.relationship") == "CastFor") &
    (F.col("b.Type") == "Question") &
    (F.col("e2.relationship") == "Links") &
    (F.col("c.Type") == "Question") &
    (F.col("e3.relationship") == "CastFor") &
    (F.col("d.Type") == "Vote")
)

print(f"Linked vote paths: {linked_vote_paths.count():,}")

In [ ]:
# Count votes per question on each end of the link
b_vote_counts = linked_vote_paths.select("a", "b").distinct().groupBy("b").count()
c_vote_counts = linked_vote_paths.select("c", "d").distinct().groupBy("c").count()

# Join and compute correlation
linked_vote_counts = (
    linked_vote_paths.filter((F.col("a.VoteTypeId") == 2) & (F.col("d.VoteTypeId") == 2))
    .select("b", "c")
    .join(b_vote_counts, on="b", how="inner")
    .withColumnRenamed("count", "b_count")
    .join(c_vote_counts, on="c", how="inner")
    .withColumnRenamed("count", "c_count")
)

correlation = linked_vote_counts.stat.corr("b_count", "c_count")
print(f"Vote correlation between linked questions: {correlation:.4f}")

## Conclusion

We learned to use GraphFrames to find network motifs in a property graph. We combined graph and relational queries to find complex patterns, and used property-level aggregation to form *property graph motifs*. Motif finding in GraphFrames is a powerful technique for exploring and understanding complex networks.

In [ ]:
spark.stop()